In [12]:
pip install adjustText

Note: you may need to restart the kernel to use updated packages.


In [15]:
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


# screening the genomic, RNA-seq, immune and TF features against MTAP status

df = pd.read_csv(
    '/home/rt334/Downloads/MEDUSA_Master_ULTIMATE.csv',
    low_memory=False
)

mtap_col = 'DriverLoss 9p21 (CDKN2A,MTAP,IFNA)'
df[mtap_col] = pd.to_numeric(df[mtap_col], errors='coerce')


# genomic features used in the screen

genomic_cols = [
    c for c in df.columns
    if c.startswith(
        ('Mutation ', 'Loss ', 'Gain ', 'DriverLoss', 'DriverGain')
    )
    and c != mtap_col
]

genomic_cols += [
    'Clonal TMB',
    'Subclonal TMB',
    'ScarHRD LOH score',
    'ScarHRD TAI score',
    'ScarHRD LST score',
    'ScarHRD sum score'
]


# leaving out chromosome 9p/9q arm calls because they overlap
# closely with the region containing the MTAP loss variable

circular_cols = [
    'Loss 9p',
    'Gain 9p',
    'Loss 9q',
    'Gain 9q'
]

genomic_cols = [
    c for c in genomic_cols
    if c not in circular_cols
]


# collecting the RNA-seq, immune and transcription-factor features

rna_cols = [
    c for c in df.columns
    if c.startswith(
        (
            'ssGSEA:',
            'RNA:',
            'TRUST4:',
            'Cibersort:',
            'ConsensusTME:',
            'xCell:',
            'decoupleR:'
        )
    )
]


# combining the feature lists and removing any duplicates

feature_cols = list(
    dict.fromkeys(genomic_cols + rna_cols)
)

print(
    f'Testing {len(feature_cols)} features against MTAP status'
)


# testing each feature between MTAP-intact and MTAP-loss patients

results = []

for feat in feature_cols:

    vals = pd.to_numeric(
        df[feat],
        errors='coerce'
    )

    g0 = vals[df[mtap_col] == 0].dropna()  # MTAP intact
    g1 = vals[df[mtap_col] == 1].dropna()  # MTAP loss

    # skipping features with too few patients in either group
    if len(g0) < 5 or len(g1) < 5:
        continue

    try:
        u, p = mannwhitneyu(
            g0,
            g1
        )
    except ValueError:
        continue


    # calculating rank-biserial correlation as an effect-size estimate

    n0, n1 = len(g0), len(g1)

    effect = 1 - (2 * u) / (n0 * n1)

    results.append({
        'Feature': feat,
        'n0': n0,
        'n1': n1,
        'p_value': p,
        'effect_size': effect
    })


# putting all of the results into one table

res_df = pd.DataFrame(results)


# correcting for multiple testing using Benjamini-Hochberg FDR

res_df['FDR'] = multipletests(
    res_df['p_value'],
    method='fdr_bh'
)[1]

res_df = res_df.sort_values(
    'p_value'
)

res_df.to_csv(
    'table_mtap_full_univariate.csv',
    index=False
)


# summary of the screen

print(
    f'\nTotal features tested: {len(res_df)}'
)

print(
    'Nominally significant (p < 0.05): '
    f"{(res_df['p_value'] < 0.05).sum()}"
)

print(
    'FDR-significant (FDR < 0.05): '
    f"{(res_df['FDR'] < 0.05).sum()}"
)

print('\nTop 15:')

print(
    res_df.head(15)[
        [
            'Feature',
            'n0',
            'n1',
            'p_value',
            'FDR',
            'effect_size'
        ]
    ].to_string(index=False)
)

print(
    '\nSaved: table_mtap_full_univariate.csv'
)

Testing 957 features against MTAP status

Total features tested: 957
Nominally significant (p < 0.05): 119
FDR-significant (FDR < 0.05): 14

Top 15:
                            Feature  n0  n1  p_value      FDR  effect_size
                  ScarHRD LOH score  56  80 0.000001 0.001176     0.488839
                           Loss 14q  56  80 0.000007 0.001611     0.367857
            DriverLoss 14q (RAD51B)  56  80 0.000007 0.001611     0.367857
                           Loss 22q  56  80 0.000008 0.001611     0.323214
               DriverLoss 22q (NF2)  56  80 0.000008 0.001611     0.323214
                    decoupleR:HCFC1  46  70 0.000070 0.010073     0.437888
                         Loss ratio  56  80 0.000074 0.010073     0.400446
Cibersort:Dendritic cells activated  46  70 0.000277 0.029931    -0.370186
                     decoupleR:MTF2  46  70 0.000281 0.029931    -0.400000
                     decoupleR:PAX5  46  70 0.000443 0.040846    -0.386957
                    decoup

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from adjustText import adjust_text


# loads the full MTAP univariate results
df = pd.read_csv('/home/rt334/Independent/Python_coded/Completed_Comments/table_mtap_full_univariate.csv')

# converts p-values to -log10 scale for the volcano plot
df['neg_log10_p'] = -np.log10(df['p_value'])

# group features by significance level
df['category'] = 'Not significant'
df.loc[df['p_value'] < 0.05, 'category'] = 'Nominal p < 0.05'
df.loc[df['FDR'] < 0.05, 'category'] = 'FDR < 0.05'


# colours used for each significance group
colours = {
    'Not significant': 'lightgrey',
    'Nominal p < 0.05': 'teal',
    'FDR < 0.05': 'firebrick'
}


fig, ax = plt.subplots(figsize=(12, 9))


# plots each significance group separately
for category in [
    'Not significant',
    'Nominal p < 0.05',
    'FDR < 0.05'
]:
    subset = df[df['category'] == category]

    if category == 'Not significant':
        point_size = 25
        point_alpha = 0.7

    elif category == 'Nominal p < 0.05':
        point_size = 45
        point_alpha = 0.9

    else:
        point_size = 65
        point_alpha = 0.9

    ax.scatter(
        subset['effect_size'],
        subset['neg_log10_p'],
        color=colours[category],
        s=point_size,
        alpha=point_alpha,
        label=f'{category} (n={len(subset)})',
        zorder=3
    )


# add some extra space around the plot so labels do not get cut off
ax.set_ylim(
    -0.3,
    df['neg_log10_p'].max() * 1.2
)

ax.set_xlim(
    df['effect_size'].min() * 1.25,
    df['effect_size'].max() * 1.25
)


# only label features that remain significant after FDR correction
significant = df[df['FDR'] < 0.05].copy()


# merge labels where the same event is represented twice
merge_map = {
    'Loss 14q':
        'Loss 14q / DriverLoss 14q (RAD51B)',

    'DriverLoss 14q (RAD51B)':
        None,

    'Loss 22q':
        'Loss 22q / DriverLoss 22q (NF2)',

    'DriverLoss 22q (NF2)':
        None
}

significant['label'] = significant['Feature'].map(
    lambda feature: merge_map.get(feature, feature)
)

significant = significant.dropna(
    subset=['label']
)


# shorten a few labels so the figure stays readable
significant['label'] = (
    significant['label']
    .str.replace('decoupleR:', 'TF: ', regex=False)
    .str.replace(' (RAD51B)', '', regex=False)
    .str.replace(' (NF2)', '', regex=False)
)


# manually move the labels that sit too close together
manual_offsets = {
    'decoupleR:STOX1': (0.02, 0.35),
    'decoupleR:TBX2': (0.06, -0.15),
    'ScarHRD sum score': (0.02, -0.45),
    'Mutation NF2': (-0.08, 0.05)
}


automatic_labels = []

for _, row in significant.iterrows():

    if row['Feature'] in manual_offsets:
        dx, dy = manual_offsets[row['Feature']]

        ax.annotate(
            row['label'],
            xy=(
                row['effect_size'],
                row['neg_log10_p']
            ),
            xytext=(
                row['effect_size'] + dx,
                row['neg_log10_p'] + dy
            ),
            fontsize=8.5,
            arrowprops={
                'arrowstyle': '-',
                'color': 'grey',
                'lw': 0.6
            }
        )

    else:
        automatic_labels.append(
            ax.text(
                row['effect_size'],
                row['neg_log10_p'],
                row['label'],
                fontsize=8.5
            )
        )


# automatically spreads out the remaining labels
adjust_text(
    automatic_labels,
    ax=ax,
    arrowprops={
        'arrowstyle': '-',
        'color': 'grey',
        'lw': 0.7
    },
    expand_points=(2.2, 2.2),
    expand_text=(1.6, 1.6),
    force_points=(0.6, 0.6),
    force_text=(0.6, 0.6),
    only_move={
        'points': 'xy',
        'text': 'xy'
    }
)


# show the nominal p = 0.05 threshold
ax.axhline(
    -np.log10(0.05),
    color='grey',
    linestyle=':',
    linewidth=0.8
)


# arrows below the x-axis show the direction of the effect
# positive values are higher in MTAP-loss, negative values in MTAP-intact
arrow_y = -0.14
label_y = -0.19

ax.annotate(
    '',
    xy=(0.02, arrow_y),
    xytext=(0.485, arrow_y),
    xycoords='axes fraction',
    annotation_clip=False,
    arrowprops={'arrowstyle': '-|>', 'color': 'navy', 'lw': 1.4}
)

ax.text(
    0.25, label_y,
    'Higher in MTAP-intact',
    transform=ax.transAxes,
    ha='center',
    fontsize=9.5,
    style='italic',
    color='navy'
)

ax.annotate(
    '',
    xy=(0.98, arrow_y),
    xytext=(0.515, arrow_y),
    xycoords='axes fraction',
    annotation_clip=False,
    arrowprops={'arrowstyle': '-|>', 'color': 'darkorange', 'lw': 1.4}
)

ax.text(
    0.75, label_y,
    'Higher in MTAP-loss',
    transform=ax.transAxes,
    ha='center',
    fontsize=9.5,
    style='italic',
    color='darkorange'
)

ax.set_xlabel(
    'Effect size '
    '(rank-biserial correlation, MTAP loss vs intact)'
)

ax.set_ylabel(
    r'$-\log_{10}(p\mathrm{-value})$'
)

ax.set_title(
    'MTAP status associations across genomic, immune and '
    'transcription-factor features\n'
    '(Mann–Whitney U tests with FDR correction)'
)

ax.legend(
    loc='upper left',
    fontsize=9,
    frameon=False
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

# save a high-resolution copy for the dissertation
plt.savefig(
    'figure_mtap_full_volcano.png',
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)

plt.close()

print('Saved figure_mtap_full_volcano.png')

Looks like you are using a tranform that doesn't support FancyArrowPatch, using ax.annotate instead. The arrows might strike through texts. Increasing shrinkA in arrowprops might help.


Saved figure_mtap_full_volcano.png


In [27]:
import pandas as pd
import math
import matplotlib.pyplot as plt

# load the MTAP pathway results
df = pd.read_csv('/home/rt334/Independent/table_mtap_vs_pathways.csv')

# take the 12 pathways with the lowest uncorrected p-values
top = (
    df.nsmallest(12, 'p-value')
    .sort_values('p-value', ascending=False)
    .copy()
)

# convert the p-values to -log10 for plotting
top['neg_log10_p'] = top['p-value'].apply(
    lambda value: -math.log10(value)
)

# shorten the pathway names so they are easier to read
top['label'] = (
    top['Pathway']
    .str.replace('HALLMARK_', '', regex=False)
    .str.replace('_', ' ', regex=False)
)

minimum_fdr = df['FDR'].min()

fig, ax = plt.subplots(figsize=(8, 6))

# plot the pathway rankings
ax.hlines(
    y=top['label'],
    xmin=0,
    xmax=top['neg_log10_p'],
    color='slategrey',
    linewidth=2
)
ax.scatter(
    top['neg_log10_p'],
    top['label'],
    color='midnightblue',
    s=80,
    zorder=3
)

ax.set_xlabel(
    r'$-\log_{10}(p\mathrm{-value})$ (uncorrected)'
)
ax.set_ylabel(
    'Hallmark pathway'
)
ax.set_title(
    'Highest-ranked Hallmark pathways associated with MTAP status',
    fontsize=12,
    fontweight='bold',
    pad=12
)

# remove the top and right borders to keep the figure clean
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(
    axis='both',
    labelsize=9
)

# p-values show the strength of the association rather than its direction
# the MTAP-loss label is based on the direction seen in the underlying results

# add the MTAP-loss direction marker on the right
ax.annotate(
    '',
    xy=(1.02, 0.98), xycoords='axes fraction',
    xytext=(1.02, 0.02), textcoords='axes fraction',
    annotation_clip=False,
    arrowprops={'arrowstyle': '-|>', 'color': 'indianred', 'lw': 2.5}
)
ax.text(
    1.05, 0.5,
    'Enriched in MTAP-deleted tumours',
    transform=ax.transAxes,
    rotation=90, ha='left', va='center',
    fontsize=11, fontweight='bold', style='italic', color='indianred'
)

plt.tight_layout()
plt.savefig(
    'figure_ssgsea_ranked.png',
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)
plt.close()

print('Saved figure_ssgsea_ranked.png')
print(
    f'Minimum FDR across all pathways: {minimum_fdr:.3f}'
)

Saved figure_ssgsea_ranked.png
Minimum FDR across all pathways: 0.056


In [26]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns


# running PCA on the available multi-omic features and colouring samples by MTAP status

df = pd.read_csv(
    '/home/rt334/Downloads/MEDUSA_Master_ULTIMATE.csv',
    low_memory=False
)

mtap_col = 'DriverLoss 9p21 (CDKN2A,MTAP,IFNA)'

df[mtap_col] = pd.to_numeric(
    df[mtap_col],
    errors='coerce'
)


# selecting numeric features while leaving out the MTAP target
# and columns used only as identifiers or availability flags

excluded_columns = [
    'MEDUSA_Subject_ID',
    mtap_col,
    'Has_Spatial_Protein_Data',
    'Has_Spatial_RNA_Data'
]

numeric_df = df.select_dtypes(
    include=['number']
).drop(
    columns=[
        column
        for column in excluded_columns
        if column in df.columns
    ],
    errors='ignore'
)


# keeping features with data available for at least half of the cohort

completeness = numeric_df.notna().mean()

numeric_df = numeric_df[
    completeness[completeness >= 0.5].index
]

print(f'Number of features used for PCA: {numeric_df.shape[1]}')


# replacing missing values, scaling the features and calculating the first two components

imputer = SimpleImputer(
    strategy='mean'
)

scaler = StandardScaler()

imputed_values = imputer.fit_transform(
    numeric_df
)

scaled_values = scaler.fit_transform(
    imputed_values
)

pca = PCA(
    n_components=2
)

principal_components = pca.fit_transform(
    scaled_values
)

variance_explained = pca.explained_variance_ratio_


# putting the PCA coordinates and MTAP groups into one dataframe for plotting

plot_df = pd.DataFrame(
    principal_components,
    columns=['PC1', 'PC2'],
    index=df.index
)

plot_df['MTAP_status'] = df[mtap_col].map({
    0: 'Intact',
    1: 'Lost'
})

plot_df = plot_df.dropna(
    subset=['MTAP_status']
)


# comparing PC1 scores between the two MTAP groups as an exploratory check

pc1_intact = plot_df.loc[
    plot_df['MTAP_status'] == 'Intact',
    'PC1'
]

pc1_lost = plot_df.loc[
    plot_df['MTAP_status'] == 'Lost',
    'PC1'
]

u_statistic, p_value = mannwhitneyu(
    pc1_intact,
    pc1_lost,
    alternative='two-sided'
)

print(
    'Mann–Whitney U test comparing PC1 by MTAP status: '
    f'p = {p_value:.3f}'
)


# plotting the first two principal components

fig, ax = plt.subplots(
    figsize=(7.5, 6)
)

sns.scatterplot(
    data=plot_df,
    x='PC1',
    y='PC2',
    hue='MTAP_status',
    hue_order=['Intact', 'Lost'],
    palette={
        'Intact': 'steelblue',
        'Lost': 'darkorange'
    },
    s=90,
    alpha=0.85,
    edgecolor='white',
    linewidth=0.6,
    ax=ax
)

ax.set_title(
    'Principal component analysis of the multi-omic feature space '
    'by MTAP status',
    fontsize=12,
    fontweight='bold',
    pad=12
)

ax.set_xlabel(
    f'PC1 ({variance_explained[0] * 100:.1f}% variance explained)'
)

ax.set_ylabel(
    f'PC2 ({variance_explained[1] * 100:.1f}% variance explained)'
)

ax.legend(
    title='MTAP status',
    loc='best',
    frameon=False
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

plt.savefig(
    'figure_pca_mtap_status.png',
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)

plt.close()

print('Saved figure_pca_mtap_status.png')

Number of features used for PCA: 1154
Mann–Whitney U test comparing PC1 by MTAP status: p = 0.619
Saved figure_pca_mtap_status.png


In [1]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import seaborn as sns


# running t-SNE on the available multi-omic features and colouring samples by MTAP status

df = pd.read_csv(
    '/home/rt334/Downloads/MEDUSA_Master_ULTIMATE.csv',
    low_memory=False
)

mtap_col = 'DriverLoss 9p21 (CDKN2A,MTAP,IFNA)'

df[mtap_col] = pd.to_numeric(
    df[mtap_col],
    errors='coerce'
)


# selecting numeric features while leaving out the MTAP target
# and columns used only as identifiers or availability flags

excluded_columns = [
    'MEDUSA_Subject_ID',
    mtap_col,
    'Has_Spatial_Protein_Data',
    'Has_Spatial_RNA_Data'
]

numeric_df = df.select_dtypes(
    include=['number']
).drop(
    columns=[
        column
        for column in excluded_columns
        if column in df.columns
    ],
    errors='ignore'
)


# keeping features with data available for at least half of the cohort

completeness = numeric_df.notna().mean()

numeric_df = numeric_df[
    completeness[completeness >= 0.5].index
]

print(f'Number of features available before t-SNE: {numeric_df.shape[1]}')


# replacing missing values and scaling the features

imputer = SimpleImputer(
    strategy='mean'
)

scaler = StandardScaler()

imputed_values = imputer.fit_transform(
    numeric_df
)

scaled_values = scaler.fit_transform(
    imputed_values
)


# reducing the high-dimensional feature space before t-SNE
# this helps remove noise and makes t-SNE more stable and efficient

n_pca_components = min(
    50,
    scaled_values.shape[0] - 1,
    scaled_values.shape[1]
)

pca = PCA(
    n_components=n_pca_components,
    random_state=42
)

pca_values = pca.fit_transform(
    scaled_values
)

print(
    f'Number of PCA dimensions passed to t-SNE: '
    f'{n_pca_components}'
)


# running t-SNE to produce a two-dimensional representation

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate='auto',
    init='pca',
    max_iter=1000,
    random_state=42
)

tsne_components = tsne.fit_transform(
    pca_values
)


# putting the t-SNE coordinates and MTAP groups into one dataframe for plotting

plot_df = pd.DataFrame(
    tsne_components,
    columns=['tSNE1', 'tSNE2'],
    index=df.index
)

plot_df['MTAP_status'] = df[mtap_col].map({
    0: 'Intact',
    1: 'Lost'
})

plot_df = plot_df.dropna(
    subset=['MTAP_status']
)

print(
    f"MTAP-intact patients plotted: "
    f"{(plot_df['MTAP_status'] == 'Intact').sum()}"
)

print(
    f"MTAP-lost patients plotted: "
    f"{(plot_df['MTAP_status'] == 'Lost').sum()}"
)


# plotting the two-dimensional t-SNE embedding

fig, ax = plt.subplots(
    figsize=(7.5, 6)
)

sns.scatterplot(
    data=plot_df,
    x='tSNE1',
    y='tSNE2',
    hue='MTAP_status',
    hue_order=['Intact', 'Lost'],
    palette={
        'Intact': 'steelblue',
        'Lost': 'darkorange'
    },
    s=90,
    alpha=0.85,
    edgecolor='white',
    linewidth=0.6,
    ax=ax
)

ax.set_title(
    't-SNE of the multi-omic feature space by MTAP status',
    fontsize=12,
    fontweight='bold',
    pad=12
)

ax.set_xlabel(
    't-SNE 1'
)

ax.set_ylabel(
    't-SNE 2'
)

ax.legend(
    title='MTAP status',
    loc='best',
    frameon=False
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

plt.savefig(
    'figure_tsne_mtap_status.png',
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)

plt.close()

print('Saved figure_tsne_mtap_status.png')

Number of features available before t-SNE: 1154
Number of PCA dimensions passed to t-SNE: 50
MTAP-intact patients plotted: 56
MTAP-lost patients plotted: 80
Saved figure_tsne_mtap_status.png
